In [ ]:
#!pip install --upgrade transformers datasets scikit-learn==1.1.3 pandas numpy -q
!pip install transformers datasets scikit-learn pandas numpy

In [ ]:
!git clone https://github.com/ShareChatAI/MACD.git

Cloning into 'MACD'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 139 (delta 21), reused 0 (delta 0), pack-reused 92 (from 1)
Receiving objects: 100% (139/139), 33.22 MiB | 7.17 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [ ]:
import pandas as pd

train_path = "/content/MACD/dataset/telugu_train.csv"
val_path   = "/content/MACD/dataset/telugu_val.csv"
test_path  = "/content/MACD/dataset/telugu_test.csv"

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (18000, 2)
Validation: (6000, 2)
Test: (6000, 2)


,label,text
0,1,"కరోనా డార్లింగ్,మా నుండి నువ్వు విడిపోయిన, మమ్..."
1,0,విడు ఈపాటికి ఎంత మంది అమ్మయిలని దేగిండో
2,0,నీ sandulu కావాలి
3,0,నా కొడకా పవన కళ్యాణ్ వెంట్రుక కూడా పికలేవు
4,1,నా పరిస్థితి అంతే


In [ ]:
import re

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"[^a-zA-Z\u0C00-\u0C7F\s]", "", text)  # keep Telugu + English
    return text

train_df['text'] = train_df['text'].apply(preprocess)
val_df['text']   = val_df['text'].apply(preprocess)
test_df['text']  = test_df['text'].apply(preprocess)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[['text','label']])
val_dataset   = Dataset.from_pandas(val_df[['text','label']])
test_dataset  = Dataset.from_pandas(test_df[['text','label']])

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize(example):
    return tokenizer(example['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset   = val_dataset.map(tokenize, batched=True)
test_dataset  = test_dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3
)


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.640319
1000,0.622076
1500,0.577267
2000,0.495798
2500,0.478973
3000,0.413390


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3375, training_loss=0.5197695538556134, metrics={'train_runtime': 1784.2269, 'train_samples_per_second': 30.265, 'train_steps_per_second': 1.892, 'total_flos': 3551999247360000.0, 'train_loss': 0.5197695538556134, 'epoch': 3.0})

In [ ]:
val_results = trainer.evaluate()
print("Validation Results:", val_results)

Validation Results: {'eval_loss': 0.3733983635902405, 'eval_accuracy': 0.8345, 'eval_f1': 0.8261858918256608, 'eval_precision': 0.8392603129445235, 'eval_recall': 0.8135125818683213, 'eval_runtime': 46.6796, 'eval_samples_per_second': 128.536, 'eval_steps_per_second': 8.033, 'epoch': 3.0}


In [ ]:
val_results = trainer.evaluate()
print("Validation Results:", val_results)

Validation Results: {'eval_loss': 0.3733983635902405, 'eval_accuracy': 0.8345, 'eval_f1': 0.8261858918256608, 'eval_precision': 0.8392603129445235, 'eval_recall': 0.8135125818683213, 'eval_runtime': 44.432, 'eval_samples_per_second': 135.038, 'eval_steps_per_second': 8.44, 'epoch': 3.0}


In [ ]:
test_results = trainer.evaluate(test_dataset)
print("Test Results:", test_results)

Test Results: {'eval_loss': 0.3943713903427124, 'eval_accuracy': 0.8318333333333333, 'eval_f1': 0.8173094332790151, 'eval_precision': 0.8306956201693044, 'eval_recall': 0.8043478260869565, 'eval_runtime': 46.2432, 'eval_samples_per_second': 129.749, 'eval_steps_per_second': 8.109, 'epoch': 3.0}


In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

preds_output = trainer.predict(test_dataset)
y_pred = np.argmax(preds_output.predictions, axis=1)
y_true = preds_output.label_ids

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[2734  460]
 [ 549 2257]]


In [ ]:
trainer.save_model("/content/macd_telugu_model")
tokenizer.save_pretrained("/content/macd_telugu_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/macd_telugu_model/tokenizer_config.json',
 '/content/macd_telugu_model/tokenizer.json')

In [ ]:
import torch

def predict(text):
    if isinstance(text,str):
      text = [text]
    else:
      text = [str(t) for t in text]
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
      outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    return probs.detach().cpu().numpy() # Move to CPU before converting to numpy

print(predict("లంజ"))

[[0.9802372  0.01976283]]


In [ ]:
!pip install shap -q

In [ ]:
import shap

explainer = shap.Explainer(predict, tokenizer)
#shap_values = explainer(["నువ్వు ఒక లంజ"])
#shap.plots.text(shap_values)
test_two = explainer(["అతను మంచి లంజ"])
shap.plots.text(test_two)
print(predict("అతను మంచి లంజ"))

[[0.97777987 0.0222202 ]]
